In [ ]:
from netCDF4 import Dataset
import numpy as np
import os
from matplotlib import pyplot as plt
import fnmatch
import datetime

In [ ]:
def find_files(directory, pattern, maxdepth=None):
    flist = []
    for root, dirs, files in os.walk(directory):
        for basename in files:
            if fnmatch.fnmatch(basename, pattern):
                filename = os.path.join(root, basename)
                filename = filename.replace('\\\\', os.sep)
                if maxdepth is None:
                    flist.append(filename)
                else:
                    if filename.count(os.sep)-directory.count(os.sep) <= maxdepth:
                        flist.append(filename)
    return flist

In [ ]:
R  = 6400

def distance(lat1, lon1, lat2, lon2):
    lat_av = np.deg2rad(lat1 - lat2)/2
    lon_av = np.deg2rad(lon1 - lon2)/2
    dd = np.sin(lat_av)**2 + np.cos(np.deg2rad(lat2)) * np.cos(np.deg2rad(lat1)) * np.sin(lon_av)**2
    dd = 2*R*np.arcsin(np.sqrt(dd))
    return dd

In [ ]:
files = find_files('/mnt/hippocamp/DATA/satellite/SMAP_V6.0/L2C/2018/', '*.nc')
files.sort()
len(files)

In [ ]:
files[14*70]

In [ ]:
SMAP_data = Dataset(f'{files[14*70]}', 'r')

In [ ]:
SMAP_data.variables

In [ ]:
celtime = np.asarray(SMAP_data['time'])
cellat = np.asarray(SMAP_data['cellat'])
cellon = np.asarray(SMAP_data['cellon'])
time = np.array([datetime.datetime(2000, 1 , 1) + datetime.timedelta(seconds=t) for t in celtime.ravel()]).reshape(celtime.shape)
time.shape, cellat.shape, cellon.shape

In [ ]:
gland = np.asarray(SMAP_data['gland'])
fland = np.asarray(SMAP_data['fland'])
gice_est = np.asarray(SMAP_data['gice_est'])
surtep = np.asarray(SMAP_data['surtep'])
winspd = np.asarray(SMAP_data['winspd'])
windir = np.asarray(SMAP_data['windir'])
solar_flux = np.asarray(SMAP_data['solar_flux'])
sunglt = np.asarray(SMAP_data['sunglt'])
monglt = np.asarray(SMAP_data['monglt'])
tb_sur0_sic = np.asarray(SMAP_data['tb_sur0_sic'])
sss_smap = np.asarray(SMAP_data['sss_smap'])
iqc_flag = np.asarray(SMAP_data['iqc_flag'])
tb_consistency = np.asarray(SMAP_data['tb_consistency'])

print(
    gland.shape, fland.shape, gice_est.shape, surtep.shape, winspd.shape, windir.shape, solar_flux.shape,
    sunglt.shape, monglt.shape, tb_sur0_sic.shape, sss_smap.shape, iqc_flag.shape, tb_consistency.shape
)

In [ ]:
gice_est_2l = np.repeat(gice_est[:, :, np.newaxis], 2, axis=2)
surtep_2l = np.repeat(surtep[:, :, np.newaxis], 2, axis=2)
winspd_2l = np.repeat(winspd[:, :, np.newaxis], 2, axis=2)
windir_2l = np.repeat(windir[:, :, np.newaxis], 2, axis=2)
solar_flux_2l = np.repeat(solar_flux[:, :, np.newaxis], 2, axis=2)

In [ ]:
tb_sur0_sic_0 = tb_sur0_sic[:,:,:,0]
tb_sur0_sic_1 = tb_sur0_sic[:,:,:,1]
tb_sur0_sic_2 = tb_sur0_sic[:,:,:,2]
tb_sur0_sic_3 = tb_sur0_sic[:,:,:,3]

In [ ]:
print(
    gland.shape, fland.shape, gice_est_2l.shape, surtep_2l.shape, winspd_2l.shape, windir_2l.shape, solar_flux_2l.shape,
    sunglt.shape, monglt.shape, sss_smap.shape, iqc_flag.shape, tb_consistency.shape,
    tb_sur0_sic_0.shape, tb_sur0_sic_1.shape, tb_sur0_sic_2.shape, tb_sur0_sic_3.shape
)

In [ ]:
def get_problem_bits(value):
    value = int(value)
    return [i for i in range(value.bit_length()) if value & (1 << i)]

In [ ]:
vars = {
    'time': time, 'cellat': cellat, 'cellon': cellon,
    'gland': gland, 'fland': fland, 'gice_est_2l': gice_est_2l,
    'surtep_2l': surtep_2l, 'winspd_2l': winspd_2l, 'windir_2l': windir_2l,
    'solar_flux_2l': solar_flux_2l, 'sunglt': sunglt, 'monglt': monglt,
    'tb_sur0_sic_0': tb_sur0_sic_0, 'tb_sur0_sic_1': tb_sur0_sic_1, 'tb_sur0_sic_2': tb_sur0_sic_2, 'tb_sur0_sic_3': tb_sur0_sic_3,
    'sss_smap': sss_smap, 'iqc_flag': iqc_flag, 'tb_consistency': tb_consistency
}

In [ ]:
vars.keys()

In [ ]:
flat_vars = {var: vars[var].ravel() for var in vars.keys()}

In [ ]:
flat_vars